We will build the data behind the sample-lineage chart: for each track on the album, every sample it pulled from the past (`sampled` / `covered in`) and every later song it went on to influence (`was sampled in` / `remixed in`).

`get-samples.ipynb`'s scraped output (`output/samples_{album}.csv`) never captured that direction, only the sample element (drums, vocals, etc.), so it can't tell past from future. The manually curated `input/{album_slug}_samples.csv` has direction + year + element, so that's the source of truth here.

Track names in that file don't always match `album_data_viz_{album}.json` exactly (case, remix subtitles), so we re-key every row against the canonical track list produced by `join-data.ipynb` — that keeps track identity and ordering identical across all three data-viz charts.

In [1]:
import os
import re
import json
import pandas as pd

In [2]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

In [3]:
script_dir = os.path.dirname(os.path.abspath('get-samples-timeline.ipynb'))

In [4]:
def normalize(s):
    s = s.strip().lower().replace("\u2019", "'")
    s = re.sub(r'\(.*?\)', '', s)  # drop parenthetical suffixes (e.g. remix credits)
    s = re.sub(r'[.]+', ' ', s)       # ellipses/periods vary between sources
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def slugify(s):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', s.lower())).strip('_')

album_slug = slugify(f'{artist} {album}')
album_slug

'common_like_water_for_chocolate'

In [5]:
# canonical track identity + order, same source join-data.ipynb produces
album_json_path = os.path.join(
    script_dir, '..', 'albums', album, 'output', f'album_data_viz_{album}.json'
)
with open(album_json_path) as f:
    canonical_tracks = json.load(f)

canonical_by_key = {
    normalize(t['track_name']): (t['track_name'], t['track_number'])
    for t in canonical_tracks
}

In [6]:
input_csv_path = os.path.join(
    script_dir, '..', 'albums', album, 'input', f'{album_slug}_samples.csv'
)
df = pd.read_csv(input_csv_path)
df.head()

,album,album_artist,album_year,track_number,track_title,track_artist,sample_type,sample_title,sample_artist,sample_year,sample_element,sample_image_url
0,Like Water for Chocolate,Common,2000,1,The Light,Common,sampled,Open Your Eyes,Bobby Caldwell,1980,Multiple Elements,https://www.whosampled.com/static/images/media...
1,Like Water for Chocolate,Common,2000,1,The Light,Common,sampled,You're Getting a Little Too Smart,Detroit Emeralds,1973,Drums,https://www.whosampled.com/static/images/media...
2,Like Water for Chocolate,Common,2000,1,The Light,Common,sampled,Track 3 (Another Batch),J Dilla,1998,Multiple Elements,https://www.whosampled.com/static/images/redes...
3,Like Water for Chocolate,Common,2000,1,The Light,Common,was sampled in,Good Flirts,Baby Keem feat. Kendrick Lamar and Momo Boyd,2026,Vocals / Lyrics,https://www.whosampled.com/static/images/media...
4,Like Water for Chocolate,Common,2000,1,The Light,Common,was sampled in,I Love Her Again,J. Cole,2026,Vocals / Lyrics,https://www.whosampled.com/static/images/media...


In [7]:
df = df[df['album'].str.casefold() == album.casefold()].copy()
df['sample_year'] = pd.to_numeric(df['sample_year'], errors='coerce').astype('Int64')
df = df.dropna(subset=['sample_year'])

df['join_key'] = df['track_title'].map(normalize)

unmatched = set(df['join_key']) - set(canonical_by_key)
if unmatched:
    print('Unmatched sample track titles — check these manually:', unmatched)

df['track_name'] = df['join_key'].map(lambda k: canonical_by_key.get(k, (None, None))[0])
df['track_number'] = df['join_key'].map(lambda k: canonical_by_key.get(k, (None, None))[1])
df = df.dropna(subset=['track_name'])
len(df)

182

In [8]:
records = df.rename(columns={
    'sample_element': 'element',
})[[
    'track_name', 'track_number', 'sample_title', 'sample_artist',
    'sample_year', 'sample_type', 'element',
]].rename(columns={'sample_type': 'direction'})
records = records.sort_values(['track_number', 'sample_year']).to_dict('records')
records[0]

{'track_name': 'Heat',
 'track_number': 2,
 'sample_title': 'Asiko (In a Silent Mix)',
 'sample_artist': 'Tony Allen',
 'sample_year': 1999,
 'direction': 'sampled',
 'element': 'Multiple Elements'}

In [9]:
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    if pd.isna(obj):
        return None
    if hasattr(obj, 'item'):
        return obj.item()
    return obj

records = make_serializable(records)

In [10]:
output_file_path = os.path.join(
    script_dir, '..', 'albums', album, 'output', f'samples_data_viz_{album}.json'
)
with open(output_file_path, 'w') as f:
    json.dump(records, f, indent=2)

output_file_path

'/Users/binguyen/Desktop/Hip-Hop-Time-Travelers/data/src/../albums/Like Water For Chocolate/output/samples_data_viz_Like Water For Chocolate.json'